# Stanford RNA 3D Folding 2 — Submission Notebook

This notebook predicts RNA 3D structures for the
[Stanford RNA 3D Folding 2](https://www.kaggle.com/competitions/stanford-rna-3d-folding-2)
Kaggle competition using the **friendly-telegram** RNA structure prediction toolkit.

**Strategy** (RhoFold+ integration for offline Kaggle submission):
1. **One-time internet-on setup** — download RhoFold+ wheels, dependencies, and
   pretrained weights; save as Kaggle datasets for offline reuse
2. **RhoFold+ ensemble generation** — primary structure prediction via pretrained
   neural model with varied recycling iterations and dropout for diversity
3. Template matching — fallback for targets with close training homologs
4. BPP-guided coordinate refinement — per-pair-type distance targets (WC 8.5Å,
   non-WC 6–7Å, stacked 3.9Å) using LinearPartition-style BPP
5. Topology correction — fixes backbone chain breaks
6. Per-target time budget — domain segmentation for long targets (L>300),
   deeper recycling for short targets (L<150)
7. Ensemble diversification — max-min RMSD selection over RhoFold+ candidates

In [ ]:
import sys, os, time, warnings, gc
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Detect environment and paths ──────────────────────────────────────────────────────────
KAGGLE = os.path.exists('/kaggle')

# Repository code (added as a Kaggle dataset)
REPO_PATH = None
for p in ['/kaggle/input/friendly-telegram',
          '/kaggle/input/friendly-telegram/friendly-telegram',
          os.path.dirname(os.path.abspath('__file__')),
          '.']:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, 'rna_dataset.py')):
        REPO_PATH = p
        break
if REPO_PATH is None:
    raise FileNotFoundError("Repository code not found — add 'friendly-telegram' as a Kaggle dataset")
sys.path.insert(0, REPO_PATH)

# Competition data
DATA_PATH = None
for p in ['/kaggle/input/stanford-rna-3d-folding-2',
          '/kaggle/input/stanford-rna-3d-folding',
          os.path.join('.', 'data', 'stanford-rna-3d-folding-2'),
          os.path.join('.', 'data', 'stanford-rna-3d-folding')]:
    if os.path.isdir(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Competition data not found — add 'stanford-rna-3d-folding-2' as data source")

OUTPUT_DIR = '/kaggle/working' if KAGGLE else '.'

# ── Paths for RhoFold+ offline assets ────────────────────────────────────────────
WHEELS_DIR = os.environ.get('KAGGLE_WHEELS',
    '/kaggle/input/rhofold-wheels')
WEIGHTS_DIR = os.environ.get('RHOFOLD_WEIGHTS_DIR',
    '/kaggle/input/rhofold-weights')
WEIGHTS_FILE = None
for p in [
    os.path.join(WEIGHTS_DIR, 'rhofold_pretrained.pt'),
    '/kaggle/input/rhofold-weights/rhofold_pretrained.pt',
    os.path.expanduser('~/.cache/rhofold/rhofold_pretrained.pt'),
]:
    if os.path.isfile(p):
        WEIGHTS_FILE = p
        break

print(f"Repository : {REPO_PATH}")
print(f"Data       : {DATA_PATH}")
print(f"Output     : {OUTPUT_DIR}")
print(f"Weights    : {WEIGHTS_FILE or '(will download)'}")
print(f"Data files : {sorted(os.listdir(DATA_PATH))}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ONE-TIME SETUP (run with internet ON, then save outputs as Kaggle datasets)
#
# This cell downloads:
#   1. RhoFold+ Python wheel and its dependencies as .whl files
#   2. RhoFold+ pretrained weights from HuggingFace
#
# After running, upload the output directories as Kaggle datasets:
#   - /kaggle/working/wheels          → dataset "rhofold-wheels"
#   - /kaggle/working/rhofold_weights → dataset "rhofold-weights"
#
# Then in subsequent offline runs, those datasets are mounted at
# /kaggle/input/rhofold-wheels and /kaggle/input/rhofold-weights.
# ═══════════════════════════════════════════════════════════════════════════

ONE_TIME_SETUP = False  # ← Set True for the first internet-on run only

if ONE_TIME_SETUP and KAGGLE:
    import subprocess

    setup_wheels = '/kaggle/working/wheels'
    setup_weights = '/kaggle/working/rhofold_weights'

    os.makedirs(setup_wheels, exist_ok=True)
    os.makedirs(setup_weights, exist_ok=True)

    print("=== Step 1/3: Downloading RhoFold+ wheel and dependencies ===")
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'download',
        'rhofold @ git+https://github.com/ml4bio/RhoFold.git',
        '--dest', setup_wheels,
    ])
    # Also download key deps as wheels for full offline install
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'download',
        'numpy', 'scipy', 'tqdm', 'huggingface_hub', 'biopython',
        '--dest', setup_wheels,
    ])

    print("=== Step 2/3: Downloading pretrained weights ===")
    from huggingface_hub import hf_hub_download
    wt_path = hf_hub_download(
        repo_id='ml4bio/RhoFold',
        filename='rhofold_pretrained.pt',
        local_dir=setup_weights,
    )
    print(f"Weights saved to: {wt_path}")

    print("=== Step 3/3: Installing RhoFold+ for this session ===")
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--no-index', '--find-links', setup_wheels,
        'rhofold',
    ])

    print()
    print("✅ One-time setup complete!")
    print(f"   Wheels  → {setup_wheels}  ({len(os.listdir(setup_wheels))} files)")
    print(f"   Weights → {setup_weights}")
    print("   Upload both as Kaggle datasets, then set ONE_TIME_SETUP = False")

elif not ONE_TIME_SETUP:
    # ── Offline install from pre-downloaded Kaggle datasets ───────────────────
    if os.path.isdir(WHEELS_DIR) and os.listdir(WHEELS_DIR):
        import subprocess
        print(f"Installing from offline wheels: {WHEELS_DIR}")
        try:
            subprocess.check_call([
                sys.executable, '-m', 'pip', 'install',
                '--no-index', '--find-links', WHEELS_DIR,
                'rhofold',
            ], stderr=subprocess.PIPE)
            print("  RhoFold installed from offline wheels \u2713")
            print("  RhoFold installed from offline wheels ✓")
        except subprocess.CalledProcessError:
            # Wheel may not match — try adding to path instead
            for sub in os.listdir(WHEELS_DIR):
                sub_path = os.path.join(WHEELS_DIR, sub)
                if os.path.isdir(sub_path) and sub_path not in sys.path:
                    sys.path.insert(0, sub_path)
            print("  Added wheel dirs to sys.path as fallback")
    else:
        print("No offline wheels found — RhoFold must already be installed")

    print("Offline setup complete ✓")

In [ ]:
# ── Load test sequences ────────────────────────────────────────────────────
test_seq_path = os.path.join(DATA_PATH, 'test_sequences.csv')
test_df = pd.read_csv(test_seq_path)
print(f"Test dataframe: {test_df.shape}")
print(test_df.head(3))

def _detect_col(df, candidates, fallback_idx=0):
    for c in candidates:
        if c in df.columns:
            return c
    return df.columns[fallback_idx]

id_col  = _detect_col(test_df, ['target_id', 'ID', 'id'])
seq_col = _detect_col(test_df, ['sequence', 'seq'], fallback_idx=1)

test_targets = {}
for _, row in test_df.iterrows():
    test_targets[str(row[id_col])] = str(row[seq_col])

print(f"\nTest targets : {len(test_targets)}")
lengths = [len(s) for s in test_targets.values()]
print(f"Length range  : {min(lengths)} – {max(lengths)} nt")

# ── Load sample submission for output format ──────────────────────────────
sample_sub_path = os.path.join(DATA_PATH, 'sample_submission.csv')
sample_sub = pd.read_csv(sample_sub_path) if os.path.isfile(sample_sub_path) else None
if sample_sub is not None:
    print(f"\nSample submission: {sample_sub.shape}")
    print(f"Columns: {list(sample_sub.columns)}")
    print(sample_sub.head(2))
else:
    print("\nNo sample_submission.csv found — will use default format")

In [ ]:
from rna_dataset import _parse_sequences, _parse_labels

train_db = {}

# ── Sequences ──
train_seq_path = os.path.join(DATA_PATH, 'train_sequences.csv')
if os.path.isfile(train_seq_path):
    tsdf = pd.read_csv(train_seq_path)
    for tid, seq in _parse_sequences(tsdf).items():
        train_db[tid] = {'sequence': seq, 'coords': None}
    del tsdf
    print(f"Training sequences : {len(train_db)}")
else:
    print("No training sequences found")

# ── Labels (coordinates) ──
train_lab_path = os.path.join(DATA_PATH, 'train_labels.csv')
if os.path.isfile(train_lab_path):
    print("Loading training labels …")
    t0 = time.time()
    tldf = pd.read_csv(train_lab_path)
    train_coords = _parse_labels(tldf)
    for tid, coords in train_coords.items():
        if tid in train_db:
            train_db[tid]['coords'] = coords
        else:
            train_db[tid] = {'sequence': '', 'coords': coords}
    del tldf, train_coords
    gc.collect()
    print(f"  Loaded in {time.time()-t0:.1f}s")
else:
    print("No training labels found")

n_with_coords = sum(1 for v in train_db.values() if v['coords'] is not None)
print(f"\nTemplate database : {len(train_db)} sequences, {n_with_coords} with 3-D coords")

In [ ]:
# ── Warm up Numba JIT to avoid compilation overhead during prediction ─────
print("Warming up JIT …", end=" ", flush=True)
t0 = time.time()

try:
    from whitebox.partition_function import compute_bpp_linear_approx
    from whitebox.riemannian_backbone import torsion_diffusion_step
    from modules.topology_correction import TopologyCorrector
    from modules.pseudoknot import detect_pseudoknots
    from pipeline.full_pipeline import (
        _sequence_to_int8, _torsions_to_coords,
        briq_refine_c3prime, predict_interdomain_contact_restraints,
        rerank_with_consensus, detect_structural_motifs, graft_motifs,
        sample_pseudoknot_ss,
    )

    _dummy_seq = np.array([0,1,2,3,0,1,2,3,0,1], dtype=np.int8)
    _ = compute_bpp_linear_approx(_dummy_seq, len(_dummy_seq))
    _dummy_t = np.zeros((10, 7), dtype=np.float32)
    _ = torsion_diffusion_step(_dummy_t, np.float32(0.5), np.float32(10.0))
    _ = _torsions_to_coords(_dummy_t)
    _tc = TopologyCorrector()
    _ = _tc(np.random.randn(10, 3).astype(np.float32))
    print(f"done ({time.time()-t0:.1f}s)")
    HAVE_PIPELINE = True
except Exception as e:
    print(f"\n  Pipeline warm-up failed: {e}")
    HAVE_PIPELINE = False

# ── Optional module availability flags ────────────────────────────────────────
HAVE_BRIQ = False
try:
    from rna_briq_refinement import BRiQRefinement
    HAVE_BRIQ = True
    print("BRiQ refinement available ✓")
except ImportError:
    print("BRiQ refinement not available — skipping")

HAVE_HIER_ASSEMBLY = False
try:
    from rna_hierarchical_assembly import predict_interdomain_contacts, Domain
    HAVE_HIER_ASSEMBLY = True
    print("Hierarchical assembly available ✓")
except ImportError:
    print("Hierarchical assembly not available — skipping")

HAVE_ENSEMBLE_RERANKER = False
try:
    from rna_ensemble_diversity import ConsensusReranker, RNAStructure
    HAVE_ENSEMBLE_RERANKER = True
    print("Ensemble diversity reranker available ✓")
except ImportError:
    print("Ensemble diversity reranker not available — skipping")

# ── Initialize RhoFold+ runner ────────────────────────────────────────────────────
HAVE_RHOFOLD = False
rhofold_runner = None

try:
    from rhofold_runner import RhoFoldRunner
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    rhofold_runner = RhoFoldRunner(device=device, weights_path=WEIGHTS_FILE)
    HAVE_RHOFOLD = rhofold_runner.available
    if HAVE_RHOFOLD:
        print(f"RhoFold+ loaded on {device} ✓")
    else:
        print("RhoFold+ not available — will use template/ab-initio fallback")
except Exception as e:
    print(f"RhoFold+ init failed: {e} — will use fallback")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Helper functions
# ═══════════════════════════════════════════════════════════════════════════

# Minimum k-mer Jaccard score to accept a template match
TEMPLATE_SCORE_THRESHOLD = 0.08

def kmer_set(seq, k=3):
    """Return the set of k-mers for a sequence."""
    return set(seq[i:i+k] for i in range(max(0, len(seq) - k + 1)))


def kmer_similarity(seq1, seq2, k=3):
    """Jaccard similarity of k-mer sets."""
    k1, k2 = kmer_set(seq1, k), kmer_set(seq2, k)
    if not k1 or not k2:
        return 0.0
    return len(k1 & k2) / len(k1 | k2)


# ── RNA substitution matrix (RIBOSUM-like) for Needleman-Wunsch ──────────
_RNA_SUB = {
    ('A','A'): 2, ('C','C'): 2, ('G','G'): 2, ('U','U'): 2,
    ('A','G'): 1, ('G','A'): 1, ('C','U'): 1, ('U','C'): 1,  # transitions
    ('A','C'): -1, ('C','A'): -1, ('A','U'): -1, ('U','A'): -1,
    ('G','C'): -1, ('C','G'): -1, ('G','U'): 0, ('U','G'): 0,  # wobble
}
_GAP_OPEN = -2
_GAP_EXTEND = -1


def nw_align_score(seq1, seq2):
    """Needleman-Wunsch global alignment score with RNA substitution matrix.

    Returns a normalised score in [0, 1] where 1 = perfect match.
    """
    n, m = len(seq1), len(seq2)
    if n == 0 or m == 0:
        return 0.0

    # DP with affine gap penalties
    dp = np.zeros((n + 1, m + 1), dtype=np.float32)
    for i in range(1, n + 1):
        dp[i, 0] = _GAP_OPEN + (i - 1) * _GAP_EXTEND
    for j in range(1, m + 1):
        dp[0, j] = _GAP_OPEN + (j - 1) * _GAP_EXTEND

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Default mismatch penalty of -2 for unknown nucleotide pairs (e.g. N)
            sub = _RNA_SUB.get((seq1[i-1], seq2[j-1]), -2)
            dp[i, j] = max(
                dp[i-1, j-1] + sub,
                dp[i-1, j] + _GAP_EXTEND,
                dp[i, j-1] + _GAP_EXTEND,
            )

    # Normalise: max possible score = 2 * min(n, m)
    max_score = 2.0 * min(n, m)
    if max_score <= 0:
        return 0.0
    return max(0.0, float(dp[n, m]) / max_score)


def find_best_template(test_seq, train_db, k=3):
    """Return (target_id, score) for the best training template.

    Uses both k-mer Jaccard and Needleman-Wunsch alignment, taking the
    better of the two for each candidate.
    """
    best_tid, best_score = None, 0.0
    tlen = len(test_seq)
    for tid, info in train_db.items():
        if info['coords'] is None or not info['sequence']:
            continue
        slen = len(info['sequence'])
        ratio = min(tlen, slen) / max(tlen, slen) if max(tlen, slen) > 0 else 0
        if ratio < 0.25:
            continue

        # k-mer Jaccard (fast)
        kmer_score = kmer_similarity(test_seq, info['sequence'], k)
        kmer_weighted = kmer_score * (0.5 + 0.5 * ratio)

        # Needleman-Wunsch (more accurate, only for plausible candidates)
        nw_score = 0.0
        # Only run O(n*m) NW for plausible candidates (k-mer > 0.02) and
        # sequences short enough to align quickly (<=600 nt)
        if kmer_weighted > 0.02 and max(tlen, slen) <= 600:
            nw_score = nw_align_score(test_seq, info['sequence'])
            nw_score *= (0.5 + 0.5 * ratio)

        score = max(kmer_weighted, nw_score)
        if score > best_score:
            best_score, best_tid = score, tid
    return best_tid, best_score


def resize_coords_arclength(coords, target_len):
    """Resize a (L_src, 3) coordinate array to (target_len, 3) via arc-length
    parameterised interpolation.  Preserves the overall backbone shape."""
    src_len = coords.shape[0]
    if src_len == target_len:
        return coords.copy().astype(np.float32)
    if src_len < 2:
        return np.zeros((target_len, 3), dtype=np.float32)

    # Cumulative arc length along the backbone
    diffs = np.diff(coords.astype(np.float64), axis=0)
    seg_len = np.sqrt(np.sum(diffs ** 2, axis=1))
    cumulative_arclength = np.concatenate([[0.0], np.cumsum(seg_len)])
    total = cumulative_arclength[-1]
    if total < 1e-8:
        return np.tile(coords[0].astype(np.float32), (target_len, 1))
    cumulative_arclength /= total          # normalise to [0, 1]
    t = np.linspace(0.0, 1.0, target_len)  # target positions
    out = np.zeros((target_len, 3), dtype=np.float32)
    for d in range(3):
        out[:, d] = np.interp(t, cumulative_arclength,
                               coords[:, d].astype(np.float64)).astype(np.float32)
    return out


def fill_nan_coords(coords_2d):
    """Replace NaN positions by linear interpolation along the backbone."""
    c = coords_2d.copy()
    for d in range(3):
        col = c[:, d]
        valid = ~np.isnan(col)
        if valid.sum() < 2:
            col[:] = 0.0
        elif not valid.all():
            col[~valid] = np.interp(np.where(~valid)[0],
                                     np.where(valid)[0],
                                     col[valid])
        c[:, d] = col
    return c


# ── Structural motif detection ─────────────────────────────────────────────
# Known GNRA tetraloop sequences
_GNRA_PATTERNS = {'GAAA', 'GAGA', 'GCAA', 'GUAA',
                  'GACA', 'GGAA', 'GAUA', 'GCGA',
                  'GUGA', 'GCUA', 'GGGA'}


def detect_motifs_in_sequence(sequence, ss_pairs=None):
    """Detect known RNA structural motifs (GNRA tetraloops, kink-turns).

    Parameters
    ----------
    sequence : str
        RNA sequence string.
    ss_pairs : list of tuple or None
        Base pairs from secondary structure prediction.

    Returns
    -------
    list of dict
        Each dict: {'type': str, 'start': int, 'end': int, 'sequence': str}.
    """
    L = len(sequence)
    motifs = []

    if ss_pairs is None:
        ss_pairs = []

    # Build paired position lookup
    paired = {}
    for i, j in ss_pairs:
        paired[i] = j
        paired[j] = i

    # GNRA tetraloops: 4-nt loop closed by a base pair
    for i, j in ss_pairs:
        if j - i == 5:
            loop_seq = sequence[i + 1:j]
            if len(loop_seq) == 4 and loop_seq in _GNRA_PATTERNS:
                motifs.append({
                    'type': 'GNRA_tetraloop',
                    'start': i + 1,
                    'end': j - 1,
                    'sequence': loop_seq,
                })

    # Scan for GNRA-like patterns even without SS (sequence-only detection)
    for i in range(L - 3):
        tetra = sequence[i:i + 4]
        if tetra in _GNRA_PATTERNS and i not in paired:
            motifs.append({
                'type': 'GNRA_candidate',
                'start': i,
                'end': i + 3,
                'sequence': tetra,
            })

    return motifs


print("Helpers defined ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Prediction functions
# ═══════════════════════════════════════════════════════════════════════════

def predict_from_template(test_seq, template_coords, n_conformers=5):
    """Return *n_conformers* coordinate arrays (L_test, 3) derived from a
    training template's (L_train, 5, 3) coordinate tensor."""
    L = len(test_seq)
    n_avail = template_coords.shape[1]
    conformers = []

    for c in range(n_conformers):
        src = c % n_avail
        tc = template_coords[:, src, :].copy()

        if np.any(np.isnan(tc)):
            for alt in range(n_avail):
                candidate = template_coords[:, alt, :]
                if not np.any(np.isnan(candidate)):
                    tc = candidate.copy()
                    break
            else:
                tc = fill_nan_coords(tc)

        fitted = resize_coords_arclength(tc, L)

        if c > 0:
            noise_scale = 0.3 * np.sqrt(c)
            fitted = fitted + np.random.randn(L, 3).astype(np.float32) * noise_scale

        conformers.append(fitted.astype(np.float32))

    return conformers


# ── Per-pair-type distance targets for BPP refinement ────────────────────
_WC_PAIRS = {('A','U'), ('U','A'), ('G','C'), ('C','G')}
_WOBBLE_PAIRS = {('G','U'), ('U','G')}


def _pair_target_distance(seq, i, j):
    """Return target C3'-C3' distance (angstrom) based on base-pair type."""
    pair = (seq[i], seq[j])
    if pair in _WC_PAIRS:
        return 8.5   # Watson-Crick canonical pairs
    if pair in _WOBBLE_PAIRS:
        return 7.0   # Wobble G-U pairs
    if abs(i - j) == 1:
        return 3.9   # Stacked / consecutive
    return 6.5       # Non-WC, non-consecutive


def bpp_guided_refinement(coords, seq_int8, n_steps=50, lr=0.005, sequence=None):
    """Refine (L, 3) coordinates using BPP restraints with steric repulsion
    and backbone angle constraints.

    Improvements over basic BPP refinement:
      - Per-pair-type distance targets (WC 8.5A, wobble 7.0A, stacked 3.9A)
      - Soft-sphere repulsion: penalizes non-bonded residue pairs closer than
        3.5A to prevent steric clashes
      - Backbone angle constraint: penalizes C3'-C3'-C3' angles deviating from
        ~120 deg to maintain physically realistic backbone geometry
    """
    from whitebox.partition_function import compute_bpp_linear_approx

    L = coords.shape[0]
    bpp = compute_bpp_linear_approx(seq_int8, L)

    pairs = []
    for i in range(L):
        for j in range(i + 4, L):
            if bpp[i, j] > 0.25:
                if sequence is not None:
                    target_d = _pair_target_distance(sequence, i, j)
                else:
                    target_d = 8.0
                pairs.append((i, j, float(bpp[i, j]), target_d))

    c = coords.astype(np.float64).copy()
    target_bond_dist = 3.8
    # Soft-sphere repulsion parameters
    repulsion_cutoff = 3.5  # Angstrom
    repulsion_strength = 2.0
    # Backbone angle parameters (ideal C3'-C3'-C3' angle ~120 deg)
    ideal_angle = 2.094  # ~120 deg in radians
    angle_strength = 1.0

    for _ in range(n_steps):
        forces = np.zeros_like(c)

        # BPP pair distance restraints
        for i, j, w, target_d in pairs:
            diff = c[j] - c[i]
            d = np.sqrt(np.sum(diff ** 2)) + 1e-8
            f = w * (d - target_d) * (diff / d)
            forces[i] += f * lr
            forces[j] -= f * lr

        # Backbone connectivity restraints
        for i in range(L - 1):
            diff = c[i + 1] - c[i]
            d = np.sqrt(np.sum(diff ** 2)) + 1e-8
            f = 5.0 * (d - target_bond_dist) * (diff / d)
            forces[i]     += f * lr
            forces[i + 1] -= f * lr

        # Soft-sphere steric repulsion for non-bonded pairs (vectorized)
        diffs_all = c[:, None, :] - c[None, :, :]  # (L, L, 3)
        dists_all = np.sqrt((diffs_all ** 2).sum(axis=-1)) + 1e-8  # (L, L)
        _idx = np.arange(L)
        rep_mask = (np.abs(_idx[:, None] - _idx[None, :]) >= 2) & (dists_all < repulsion_cutoff)
        if rep_mask.any():
            overlap = np.where(rep_mask, repulsion_cutoff - dists_all, 0.0)
            f_mag = repulsion_strength * overlap / dists_all
            forces += np.sum(f_mag[:, :, None] * diffs_all, axis=1) * lr

        # Backbone angle constraints (C3'-C3'-C3')
        for i in range(1, L - 1):
            v1 = c[i - 1] - c[i]
            v2 = c[i + 1] - c[i]
            n1 = np.sqrt(np.sum(v1 ** 2)) + 1e-8
            n2 = np.sqrt(np.sum(v2 ** 2)) + 1e-8
            cos_angle = np.dot(v1, v2) / (n1 * n2)
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angle = np.arccos(cos_angle)
            angle_dev = angle - ideal_angle
            f_mag = angle_strength * angle_dev * lr
            forces[i] -= f_mag * (v1 / n1 + v2 / n2)

        c += forces

    return c.astype(np.float32)


def rhofold_predict(sequence, n_conformers=5):
    """Generate conformers using RhoFold+ with varied recycling for diversity.

    Integrates:
      - BPP-guided refinement with steric/angle constraints
      - BRiQ knowledge-based energy refinement (post-BPP)
      - Motif detection and canonical geometry grafting
      - Pseudoknot-aware secondary structure for restraints
      - Inter-domain contact prediction for long targets (L>300)
      - Consensus reranking for conformer selection
    """
    from pipeline.full_pipeline import _sequence_to_int8
    from modules.topology_correction import TopologyCorrector

    L = len(sequence)
    seq_int8 = _sequence_to_int8(sequence)
    corrector = TopologyCorrector(threshold=4.5)

    # Per-target time budget
    if L < 150:
        n_candidates = min(30, max(n_conformers * 4, 15))
        recycles = [1, 2, 3, 4, 4, 4]
        bpp_steps = 80
    elif L <= 300:
        n_candidates = min(20, max(n_conformers * 3, 10))
        recycles = [1, 2, 3, 4, 4]
        bpp_steps = 50
    else:
        n_candidates = min(15, max(n_conformers * 2, 8))
        recycles = [1, 2, 4, 4]
        bpp_steps = 30

    # ── Pseudoknot-aware SS for restraints ─────────────────────────────────
    pk_ss = None
    try:
        from whitebox.partition_function import compute_bpp_linear_approx
        bpp = compute_bpp_linear_approx(seq_int8, L)
        pk_ss = sample_pseudoknot_ss(bpp)
    except Exception:
        pass

    # ── Motif detection ────────────────────────────────────────────────────
    motifs = []
    try:
        mid_ss = []
        if pk_ss:
            mid_ss = pk_ss
        motifs = detect_structural_motifs(sequence, mid_ss)
    except Exception:
        pass

    # Real pLDDT scores from RhoFold+ (populated for short targets)
    _real_plddts = None

    # Domain segmentation for long targets
    if L > 300:
        domain_size = 250
        overlap = 30
        domains = []
        start = 0
        while start < L:
            end = min(start + domain_size, L)
            domains.append((start, end))
            if end >= L:
                break
            start = end - overlap

        # Predict inter-domain contacts for orientation
        inter_domain_contacts = []
        if HAVE_HIER_ASSEMBLY:
            try:
                inter_domain_contacts = predict_interdomain_contact_restraints(
                    sequence, domains)
            except Exception:
                inter_domain_contacts = []

        candidates = []
        for ci in range(n_candidates):
            rec = recycles[ci % len(recycles)]
            domain_coords_list = []

            for d_start, d_end in domains:
                domain_seq = sequence[d_start:d_end]
                domain_coords = rhofold_runner.predict(domain_seq, n_recycles=rec)
                d_len = d_end - d_start
                domain_coords_list.append(domain_coords[:d_len])

            # Rigid-body assembly with inter-domain contact restraints
            from pipeline.full_pipeline import rigid_body_domain_assembly
            full_coords = rigid_body_domain_assembly(
                domain_coords_list, domains, L,
                inter_domain_contacts=inter_domain_contacts)
            candidates.append(full_coords)
    else:
        recycles_list = [recycles[i % len(recycles)] for i in range(n_candidates)]
        _ensemble_result = rhofold_runner.generate_ensemble(
            sequence, n_candidates=n_candidates, n_recycles_list=recycles_list,
            return_plddt=True)
        if isinstance(_ensemble_result, tuple) and len(_ensemble_result) == 2:
            candidates, _real_plddts = _ensemble_result
        else:
            candidates = _ensemble_result
            _real_plddts = None

    # BPP refinement on each candidate
    refined = []
    for ci, coords in enumerate(candidates):
        coords = corrector(coords)
        if L <= 500:
            try:
                # For first candidate, use pseudoknot restraints
                if pk_ss and ci == 0:
                    from pipeline.full_pipeline import bpp_guided_refinement as _bpp_refine
                    coords = _bpp_refine(
                        coords, seq_int8, n_steps=bpp_steps, lr=0.003,
                        sequence=sequence, extra_pair_restraints=pk_ss)
                else:
                    coords = bpp_guided_refinement(
                        coords, seq_int8, n_steps=bpp_steps, lr=0.003,
                        sequence=sequence)
                coords = corrector(coords)
            except Exception:
                pass
        refined.append(coords)

    # ── BRiQ knowledge-based refinement (post-BPP) ────────────────────────
    if HAVE_BRIQ and L <= 500:
        briq_refined = []
        for coords in refined:
            try:
                coords = briq_refine_c3prime(
                    coords, sequence, n_steps=80, verbose=False)
            except Exception:
                pass
            briq_refined.append(coords)
        refined = briq_refined

    # ── Motif grafting ────────────────────────────────────────────────────
    if motifs:
        grafted = []
        for coords in refined:
            try:
                coords, _ = graft_motifs(coords, motifs)
            except Exception:
                pass
            grafted.append(coords)
        refined = grafted

    if len(refined) <= n_conformers:
        return refined[:n_conformers]

    # ── Conformer selection: consensus reranking > pLDDT+diversity ─────────
    # Force-include pseudoknot conformer (index 0) if PK restraints were used
    _pk_force = [0] if pk_ss else None
    try:
        selected_indices = rerank_with_consensus(
            refined, sequence, n_select=n_conformers, force_include=_pk_force)
        return [refined[i] for i in selected_indices]
    except Exception:
        pass

    # Fallback: confidence-based selection using pLDDT + structural diversity
    try:
        from pipeline.full_pipeline import select_by_plddt_and_diversity
        # Use real pLDDT from RhoFold+ if available, else proxy
        plddts = None
        if _real_plddts is not None and len(_real_plddts) >= len(refined):
            plddts = _real_plddts[:len(refined)]
        if plddts is None:
            coords_stack = np.stack(refined, axis=0)  # (N, L, 3)
            centroid = coords_stack.mean(axis=0)  # (L, 3)
            plddts = []
            for c in refined:
                per_res_dist = np.sqrt(np.sum((c - centroid) ** 2, axis=1))
                plddt = 1.0 / (1.0 + per_res_dist)
                plddts.append(plddt.astype(np.float32))
        # Force-include pseudoknot conformer (index 0) if PK restraints were used
        _pk_force = [0] if pk_ss else None
        selected = select_by_plddt_and_diversity(
            refined, plddts, n_select=n_conformers, force_include=_pk_force)
        return [refined[i] for i in selected]
    except ImportError:
        pass

    # Last resort: greedy max-min RMSD diversity selection
    # Index 0 is always included (may carry pseudoknot topology)
    selected = [0]
    for _ in range(n_conformers - 1):
        best_j, best_d = 0, -1.0
        for j in range(len(refined)):
            if j in selected:
                continue
            md = min(
                np.mean(np.linalg.norm(refined[j] - refined[s], axis=1))
                for s in selected
            )
            if md > best_d:
                best_d, best_j = md, j
        selected.append(best_j)

    return [refined[i] for i in selected]


def ab_initio_predict(sequence, n_conformers=5, n_candidates=40):
    """Generate conformers ab initio via torsion-space diffusion, optionally
    refined with BPP restraints and BRiQ energy minimization.

    Integrates motif grafting and pseudoknot detection as secondary generator."""
    from pipeline.full_pipeline import _sequence_to_int8, _torsions_to_coords
    from whitebox.riemannian_backbone import torsion_diffusion_step
    from modules.topology_correction import TopologyCorrector

    L = len(sequence)
    seq_int8 = _sequence_to_int8(sequence)
    corrector = TopologyCorrector(threshold=4.5)

    # ── Motif detection ────────────────────────────────────────────────────
    motifs = []
    try:
        motifs = detect_structural_motifs(sequence, [])
    except Exception:
        pass

    candidates = []
    for i in range(n_candidates):
        torsions = np.zeros((L, 7), dtype=np.float32)
        noise_scale = np.float32(0.05 + 0.95 * i / max(n_candidates - 1, 1))
        torsions = torsion_diffusion_step(torsions, noise_scale, np.float32(10.0))
        coords = _torsions_to_coords(torsions)
        coords = corrector(coords)
        if L <= 500:
            try:
                coords = bpp_guided_refinement(coords, seq_int8, n_steps=30, lr=0.003,
                                               sequence=sequence)
                coords = corrector(coords)
            except Exception:
                pass
        candidates.append(coords)

    # ── BRiQ refinement on ab initio candidates ───────────────────────────
    if HAVE_BRIQ and L <= 300:
        briq_refined = []
        for coords in candidates:
            try:
                coords = briq_refine_c3prime(
                    coords, sequence, n_steps=50, verbose=False)
            except Exception:
                pass
            briq_refined.append(coords)
        candidates = briq_refined

    # ── Motif grafting ────────────────────────────────────────────────────
    if motifs:
        grafted = []
        for coords in candidates:
            try:
                coords_g, _ = graft_motifs(coords, motifs)
                coords = coords_g
            except Exception:
                pass
            grafted.append(coords)
        candidates = grafted

    if len(candidates) <= n_conformers:
        return candidates[:n_conformers]

    # ── Consensus reranking ───────────────────────────────────────────────
    try:
        selected_indices = rerank_with_consensus(
            candidates, sequence, n_select=n_conformers)
        return [candidates[i] for i in selected_indices]
    except Exception:
        pass

    # Fallback: greedy diversity selection
    selected = [0]
    for _ in range(n_conformers - 1):
        best_j, best_d = 0, -1.0
        for j in range(len(candidates)):
            if j in selected:
                continue
            md = min(
                np.mean(np.linalg.norm(candidates[j] - candidates[s], axis=1))
                for s in selected
            )
            if md > best_d:
                best_d, best_j = md, j
        selected.append(best_j)

    return [candidates[i] for i in selected]


print("Prediction functions defined ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Main prediction loop
# ═══════════════════════════════════════════════════════════════════════════
from modules.topology_correction import TopologyCorrector

corrector = TopologyCorrector(threshold=4.5)
predictions = {}
method_counts = {'rhofold': 0, 'template': 0, 'ab_initio': 0}

print(f"Predicting {len(test_targets)} targets …")
print(f"RhoFold+ available: {HAVE_RHOFOLD}")
overall_t0 = time.time()

for idx, (tid, seq) in enumerate(test_targets.items()):
    t0 = time.time()
    L = len(seq)

    # ── Strategy 1: RhoFold+ (primary, highest quality) ───────────────────
    if HAVE_RHOFOLD:
        try:
            conformers = rhofold_predict(seq)
            method_counts['rhofold'] += 1
            method = 'R'
        except Exception as e:
            conformers = None
            method = None
            print(f"  RhoFold+ failed for {tid}: {e}")
    else:
        conformers = None
        method = None

    # ── Strategy 2: Template-based prediction (fallback) ──────────────────
    if conformers is None:
        best_tid, best_score = find_best_template(seq, train_db)

        if best_tid is not None and best_score > TEMPLATE_SCORE_THRESHOLD and train_db[best_tid]['coords'] is not None:
            conformers = predict_from_template(seq, train_db[best_tid]['coords'])
            method_counts['template'] += 1
            method = 'T'
        elif HAVE_PIPELINE:
            conformers = ab_initio_predict(seq)
            method_counts['ab_initio'] += 1
            method = 'A'
        else:
            conformers = []
            for c in range(5):
                coords = np.zeros((L, 3), dtype=np.float32)
                for i in range(L):
                    angle = i * 0.6 + c * 0.3
                    coords[i] = [3.8 * i * np.cos(angle * 0.15),
                                  3.8 * i * np.sin(angle * 0.15),
                                  3.4 * np.sin(angle)]
                conformers.append(coords)
            method_counts['ab_initio'] += 1
            method = 'H'
        best_score = best_score if method == 'T' else 0.0
    else:
        best_score = 1.0

    # ── Ensure exactly 5 conformers ───────────────────────────────────────────
    while len(conformers) < 5:
        base = conformers[-1].copy()
        conformers.append(base + np.random.randn(L, 3).astype(np.float32) * 0.5)
    conformers = conformers[:5]

    # ── Topology correction ───────────────────────────────────────────────────
    fixed = []
    for c in conformers:
        c = np.asarray(c, dtype=np.float32)
        if c.shape != (L, 3):
            c = resize_coords_arclength(c, L) if c.shape[0] != L else c[:, :3]
        c = corrector(c)
        fixed.append(c)

    predictions[tid] = fixed

    if (idx + 1) % max(1, len(test_targets) // 10) == 0 or idx == 0:
        print(f"  [{idx+1:>4}/{len(test_targets)}] {tid:>12} L={L:>4}  "
              f"method={method}  score={best_score:.3f}  {time.time()-t0:.1f}s")

elapsed = time.time() - overall_t0
print(f"\nDone — {len(predictions)} targets in {elapsed:.1f}s")
print(f"Methods: {method_counts}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Build and save submission.csv
# ═══════════════════════════════════════════════════════════════════════════

# ── Detect expected column format ─────────────────────────────────────────
if sample_sub is not None:
    sub_columns = list(sample_sub.columns)
    id_col_name = sub_columns[0]
    coord_cols_sample = [c for c in sub_columns if c != id_col_name]
else:
    id_col_name = 'id'
    coord_cols_sample = []
    for ci in range(1, 6):
        coord_cols_sample.extend([f'x_{ci}', f'y_{ci}', f'z_{ci}'])
    sub_columns = [id_col_name] + coord_cols_sample

print(f"Submission columns ({len(sub_columns)}): {sub_columns[:6]} …")

# ── Build rows ────────────────────────────────────────────────────────────
rows = []
for tid in test_targets:
    seq = test_targets[tid]
    conformers = predictions[tid]
    L = len(seq)

    for resid in range(L):
        row = {id_col_name: f"{tid}_{resid + 1}"}
        row['resname'] = seq[resid]
        row['resid'] = resid + 1
        for ci in range(5):
            cn = ci + 1
            row[f'x_{cn}'] = round(float(conformers[ci][resid, 0]), 4)
            row[f'y_{cn}'] = round(float(conformers[ci][resid, 1]), 4)
            row[f'z_{cn}'] = round(float(conformers[ci][resid, 2]), 4)
        rows.append(row)

submission = pd.DataFrame(rows)

# ── Align to sample submission columns (handles differing col order) ──────
if sample_sub is not None:
    for col in sub_columns:
        if col not in submission.columns:
            submission[col] = 0.0
    submission = submission[sub_columns]

out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Shape: {submission.shape}")
print(submission.head(3))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cross-validation on training data (estimate expected TM-score)
# ═══════════════════════════════════════════════════════════════════════════
try:
    from metrics.tm_score import tm_score, kabsch_align

    rng = np.random.default_rng(123)
    val_tids = [tid for tid in train_db
                if train_db[tid]['coords'] is not None
                and train_db[tid]['sequence']
                and not np.any(np.isnan(train_db[tid]['coords'][:, 0, :]))]
    if len(val_tids) > 30:
        val_tids = list(rng.choice(val_tids, 30, replace=False))

    val_scores = []
    for tid in val_tids:
        true = train_db[tid]['coords'][:, 0, :]
        seq  = train_db[tid]['sequence']
        L    = len(seq)
        if true.shape[0] != L:
            continue

        pred = None

        # Try RhoFold+ first
        if HAVE_RHOFOLD:
            try:
                pred = rhofold_predict(seq, n_conformers=1)[0]
            except Exception:
                pred = None

        # Fall back to template matching (leave-one-out)
        if pred is None:
            temp_db = {k: v for k, v in train_db.items() if k != tid}
            bt, bs = find_best_template(seq, temp_db)

            if bt is not None and bs > TEMPLATE_SCORE_THRESHOLD and temp_db[bt]['coords'] is not None:
                pred = predict_from_template(seq, temp_db[bt]['coords'], n_conformers=1)[0]
            else:
                continue

        aligned = kabsch_align(pred, true)
        sc = tm_score(aligned, true)
        val_scores.append(sc)

    if val_scores:
        arr = np.array(val_scores)
        print(f"Validation TM-scores (n={len(arr)}):")
        print(f"  mean  = {arr.mean():.4f}")
        print(f"  median= {np.median(arr):.4f}")
        print(f"  >0.5  = {(arr > 0.5).sum()}/{len(arr)}")
        print(f"  >0.7  = {(arr > 0.7).sum()}/{len(arr)}")
    else:
        print("No validation targets available for scoring.")
except Exception as e:
    print(f"Validation skipped: {e}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Final submission sanity checks
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("Submission sanity checks")
print("=" * 60)

sub = pd.read_csv(os.path.join(OUTPUT_DIR, 'submission.csv'))
sub_ids = set(sub.iloc[:, 0].apply(lambda x: str(x).rsplit('_', 1)[0]))
test_ids = set(test_targets.keys())
missing = test_ids - sub_ids
print(f"Targets in test set    : {len(test_ids)}")
print(f"Targets in submission  : {len(sub_ids)}")
if missing:
    print(f"  ⚠ Missing targets: {missing}")
else:
    print("  ✓ All targets present")

expected_rows = sum(len(s) for s in test_targets.values())
print(f"Expected rows          : {expected_rows}")
print(f"Actual rows            : {len(sub)}")
assert len(sub) == expected_rows, f"Row count mismatch: {len(sub)} vs {expected_rows}"
print("  ✓ Row count correct")

coord_cols = [c for c in sub.columns if c.startswith(('x_', 'y_', 'z_'))]
vals = sub[coord_cols].values
print(f"NaN count              : {np.isnan(vals).sum()}")
print(f"Inf count              : {np.isinf(vals).sum()}")
assert np.isnan(vals).sum() == 0, "NaN values in submission!"
assert np.isinf(vals).sum() == 0, "Inf values in submission!"
print("  ✓ No NaN / Inf values")

print("\n✅  submission.csv is ready for upload!")